# Explainable AI (SHAP)

In financial applications, detecting fraud is only half the battle. If a transaction is blocked, the risk team (and sometimes the customer) needs to know **why**. 

We use **SHAP (SHapley Additive exPlanations)** to break down exactly which features contributed to a model's decision, and by how much.

In [ ]:
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.preprocessing import load_and_clean_data, preprocess_and_split
from src.feature_engineering import engineer_features

# Initialize SHAP JS visualization in the notebook
shap.initjs()

### 1. Load Data & Train Model
We will quickly train an XGBoost model using the undersampled dataset (to make SHAP explanations compute lightning fast for this demonstration).

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

data_path = '../data/raw/PS_20174392719_1491204439457_log.csv'
df = load_and_clean_data(data_path)
df_engineered = engineer_features(df)
X_train, X_test, y_train, y_test, preprocessor = preprocess_and_split(df_engineered)

under_sampler = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_train_under, y_train_under = under_sampler.fit_resample(X_train, y_train)

xgb_model = xgb.XGBClassifier(tree_method='hist', max_depth=6, n_estimators=100, random_state=42)
xgb_model.fit(X_train_under, y_train_under)
print("Model trained.")

### 2. Generate SHAP Values
TreeExplainer is optimized for tree-based models and is extremely fast.

In [ ]:
# Explain a sample of the test set to save time
X_sample = X_test.sample(2000, random_state=42)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_sample)

### 3. Global Explainability (Summary Plot)
This plot shows the overall feature importance across the entire dataset. It reveals which features drive the model the most globally.
*(Notice how our engineered feature `errorBalanceOrig` is likely near the very top!)*

In [ ]:
shap.summary_plot(shap_values, X_sample, plot_type="bar")

The beeswarm plot goes a step further by showing the *direction* of the relationship. (e.g., does a high balance error push the probability towards Fraud or Legit?)

In [ ]:
shap.summary_plot(shap_values, X_sample)

### 4. Local Explainability (Force Plot)
This is what we show to a human investigator. We take a **single specific transaction** and explain exactly why the model flagged it.

In [ ]:
# Find a known fraudulent transaction in our sample
fraud_idx = y_test.loc[X_sample.index][y_test == 1].index[0]
sample_idx = X_sample.index.get_loc(fraud_idx)

# Display the force plot for this single transaction
shap.force_plot(explainer.expected_value, shap_values[sample_idx,:], X_sample.iloc[sample_idx,:])